# OSMI Mental Health: Model Training and Hyperparameter Tuning

**Problem:** Binary classification

**Target:** `treatment` (`Yes` or `No`)

**Objective:** Predict whether a respondent has sought mental-health treatment while avoiding preprocessing and test-set leakage.

## Scope and validation protocol

- Remove exact duplicates and invalid ages.
- Use a fixed 25% stratified held-out test set.
- Run 5-fold repeated cross-validation on the training set only.
- Compare several models with leakage-safe sklearn pipelines.
- Tune only shortlisted models.
- Select the simplest model within one standard deviation of the highest tuned CV AUC.
- Tune a probability threshold using training cross-validation predictions.
- Evaluate the final model on the held-out test set once.
- Save a fitted pipeline and test artefacts for deployment.

The final scenario excludes `work_interfere`, because it may be too close to the target to be a fair standalone predictor.

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import loguniform, randint

from IPython.display import display, Markdown

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_validate,
    cross_val_predict,
    RandomizedSearchCV,
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)
from sklearn.inspection import permutation_importance

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42
TEST_SIZE = 0.25
print("Imports ready. XGBoost available:", HAS_XGBOOST)


## 1. Load and validate data

This stage is independent of the train-test split. Invalid ages are converted to missing values before the split, and median imputation happens later inside the pipeline.

In [ ]:
def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if (candidate / "data" / "processed" / "mental_health_survey_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate data/processed/mental_health_survey_cleaned.csv")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "mental_health_survey_cleaned.csv"

df = pd.read_csv(DATA_PATH)
print("Loaded shape:", df.shape)

df = df.drop_duplicates().reset_index(drop=True)

df["age"] = pd.to_numeric(df["age"], errors="coerce")
df.loc[(df["age"] < 16) | (df["age"] > 100), "age"] = np.nan

TARGET = "treatment"
FEATURES = [
    "age", "gender", "country", "self_employed", "family_history",
    "no_employees", "remote_work", "tech_company", "benefits",
    "care_options", "wellness_program", "seek_help", "anonymity",
    "leave", "mental_health_consequence", "phys_health_consequence",
    "coworkers", "supervisor", "mental_health_interview",
    "phys_health_interview", "mental_vs_physical", "obs_consequence",
]

missing_target = df[TARGET].isna().sum()
df = df.dropna(subset=[TARGET]).copy()
y = df[TARGET].map({"No": 0, "Yes": 1})
X = df[FEATURES].copy()

print("Analytical shape:", df.shape)
print("Missing targets removed:", missing_target)
print("Invalid ages:", int(X["age"].isna().sum()))
print("Missing work_interfere intentionally excluded:", df["work_interfere"].isna().sum())
print("Class distribution:")
print(y.value_counts().sort_index())


## 2. Stratified train-test split

The test set is held out before transformations and model selection. It is used once for the final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training positive rate:", round(y_train.mean(), 4))
print("Testing positive rate:", round(y_test.mean(), 4))


## 3. Leakage-safe preprocessing

- Numeric age: median imputation and standardization.
- Categorical variables: missing values become a separate category, rare categories are grouped, and values are one-hot encoded.
- The transformers are fitted inside each pipeline, never directly on the complete dataset.

In [ ]:
numeric_features = ["age"]
categorical_features = [column for column in FEATURES if column != "age"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

print("Numeric features:", numeric_features)
print("Categorical features:", len(categorical_features))


## 4. Cross-validation model comparison

This compares a baseline and representative linear, tree, boosting, and support-vector models using repeated 5-fold stratified CV.

In [ ]:
comparison_models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "LogReg": LogisticRegression(
        max_iter=3000,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
    "KNN": KNeighborsClassifier(),
    "DecisionTree": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced",
        max_depth=8,
    ),
    "RandomForest": RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced",
    ),
    "ExtraTrees": ExtraTreesClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced",
    ),
    "GradBoost": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "HistGB": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    "SVC": SVC(
        probability=True,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
}

if HAS_XGBOOST:
    comparison_models["XGBoost"] = XGBClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric="logloss",
    )

comparison_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=RANDOM_STATE,
)

comparison_rows = []
fitted_comparison = {}

for name, model in comparison_models.items():
    pipe = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", clone(model)),
    ])
    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=comparison_cv,
        scoring={
            "AUC": "roc_auc",
            "Accuracy": "accuracy",
            "F1": "f1",
        },
        n_jobs=-1,
    )
    comparison_rows.append({
        "Model": name,
        "AUC": scores["test_AUC"].mean(),
        "AUC_sd": scores["test_AUC"].std(),
        "Accuracy": scores["test_Accuracy"].mean(),
        "F1": scores["test_F1"].mean(),
    })

comparison_table = (
    pd.DataFrame(comparison_rows)
    .sort_values("AUC", ascending=False)
    .reset_index(drop=True)
)
display(comparison_table.round(4))


## 5. Hyperparameter tuning

RandomizedSearchCV tunes a shortlist of the strongest candidates on the training set only. It uses the same preprocessing pipeline and repeated cross-validation.

In [ ]:
tuning_models = {
    "LogReg": (
        LogisticRegression(
            max_iter=3000,
            random_state=RANDOM_STATE,
            class_weight="balanced",
        ),
        {
            "model__C": loguniform(0.01, 100),
            "model__penalty": ["l1", "l2"],
            "model__solver": ["liblinear"],
        },
    ),
    "RandomForest": (
        RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
        ),
        {
            "model__n_estimators": randint(200, 700),
            "model__max_depth": randint(3, 16),
            "model__min_samples_leaf": randint(1, 20),
            "model__min_samples_split": randint(2, 30),
            "model__max_features": ["sqrt", "log2", 0.7],
        },
    ),
    "ExtraTrees": (
        ExtraTreesClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
        ),
        {
            "model__n_estimators": randint(200, 700),
            "model__max_depth": randint(3, 16),
            "model__min_samples_leaf": randint(1, 20),
            "model__min_samples_split": randint(2, 30),
            "model__max_features": ["sqrt", "log2", 0.7],
        },
    ),
    "HistGB": (
        HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            "model__learning_rate": loguniform(0.01, 0.3),
            "model__max_iter": randint(100, 500),
            "model__max_leaf_nodes": randint(10, 40),
            "model__l2_regularization": loguniform(0.001, 100),
            "model__max_depth": [None, 3, 5, 8],
        },
    ),
}

if HAS_XGBOOST:
    tuning_models["XGBoost"] = (
        XGBClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            eval_metric="logloss",
        ),
        {
            "model__n_estimators": randint(150, 500),
            "model__max_depth": randint(2, 8),
            "model__learning_rate": loguniform(0.01, 0.3),
            "model__subsample": [0.7, 0.85, 1.0],
            "model__colsample_bytree": [0.7, 0.85, 1.0],
            "model__min_child_weight": randint(1, 10),
            "model__reg_lambda": loguniform(0.01, 100),
            "model__reg_alpha": loguniform(0.001, 10),
        },
    )

tuning_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=2,
    random_state=RANDOM_STATE,
)

tuning_rows = []
searches = {}

for name, (model, parameters) in tuning_models.items():
    pipe = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", clone(model)),
    ])
    search = RandomizedSearchCV(
        pipe,
        param_distributions=parameters,
        n_iter=40,
        cv=tuning_cv,
        scoring="roc_auc",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        refit=True,
    )
    search.fit(X_train, y_train)
    searches[name] = search
    tuning_rows.append({
        "Model": name,
        "Tuned CV AUC": search.best_score_,
        "Best parameters": search.best_params_,
    })

tuning_table = pd.DataFrame(tuning_rows).sort_values(
    "Tuned CV AUC",
    ascending=False,
).reset_index(drop=True)
display(tuning_table)


## 6. Select the final model

Selection favors the simplest model whose tuned CV AUC is within one standard deviation of the highest tuned AUC. The AUC standard deviation is estimated from the repeated training cross-validation of the candidate itself.

In [ ]:
top_model_name = tuning_table.loc[0, "Model"]
top_search = searches[top_model_name]

complexity_order = ["LogReg", "RandomForest", "ExtraTrees", "HistGB", "XGBoost"]
top_auc = tuning_table["Tuned CV AUC"].max()

selection_rows = []
for name in tuning_table["Model"]:
    search = searches[name]
    scores = cross_validate(
        search.best_estimator_,
        X_train,
        y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        scoring="roc_auc",
        n_jobs=-1,
    )
    sd = scores["test_score"].std()
    selection_rows.append({"Model": name, "AUC": search.best_score_, "AUC_sd": sd})

selection_table = pd.DataFrame(selection_rows).sort_values("AUC", ascending=False)
eligible = selection_table[selection_table["AUC"] >= top_auc - selection_table.iloc[0]["AUC_sd"]]
eligible["Complexity"] = eligible["Model"].map(
    {name: index for index, name in enumerate(complexity_order)}
)
selected_name = eligible.sort_values("Complexity").iloc[0]["Model"]
selected_search = searches[selected_name]
selected_model = selected_search.best_estimator_

print("Highest tuned CV AUC:", round(top_auc, 4), "by", top_model_name)
print("Selected model:", selected_name)
print("Selected parameters:", selected_search.best_params_)


## 7. Choose the probability threshold using training CV only

The default 0.5 threshold is not always optimal for F1. We choose the threshold that maximizes F1 across 5-fold out-of-fold training predictions, then apply it exactly once to the test set.

In [ ]:
threshold_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

oof_probabilities = cross_val_predict(
    selected_model,
    X_train,
    y_train,
    cv=threshold_cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

threshold_candidates = np.linspace(0.25, 0.75, 101)
threshold_scores = [
    f1_score(y_train, (oof_probabilities >= threshold).astype(int))
    for threshold in threshold_candidates
]
best_threshold_index = int(np.argmax(threshold_scores))
best_threshold = float(threshold_candidates[best_threshold_index])

print("Training-CV optimal threshold:", round(best_threshold, 4))
print("Training-CV F1:", round(max(threshold_scores), 4))


## 8. Final held-out test evaluation

The test set has not been used for training, feature selection, tuning, or threshold selection.

In [ ]:
test_probabilities = selected_model.predict_proba(X_test)[:, 1]
default_predictions = (test_probabilities >= 0.5).astype(int)
tuned_predictions = (test_probabilities >= best_threshold).astype(int)

def evaluate(y_true, y_pred, probabilities):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "AUC": roc_auc_score(y_true, probabilities),
    }

default_results = evaluate(y_test, default_predictions, test_probabilities)
tuned_results = evaluate(y_test, tuned_predictions, test_probabilities)

test_comparison = pd.DataFrame(
    [default_results, tuned_results],
    index=["Default 0.50 threshold", f"CV threshold {best_threshold:.4f}"],
)
display(test_comparison.round(4))

print("\nClassification report at the CV threshold:")
print(classification_report(
    y_test,
    tuned_predictions,
    target_names=["No", "Yes"],
    zero_division=0,
))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    tuned_predictions,
    display_labels=["No", "Yes"],
    ax=axes[0],
)
axes[0].set_title(f"{selected_name} - Confusion Matrix")
RocCurveDisplay.from_predictions(y_test, test_probabilities, ax=axes[1])
axes[1].set_title(f"{selected_name} - ROC Curve")
plt.tight_layout()
plt.show()


## 9. Bootstrap confidence interval for test AUC

A bootstrap interval quantifies uncertainty from the one held-out test sample. It does not tune the model.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n_bootstrap = 1000
bootstrap_auc = []

for _ in range(n_bootstrap):
    indices = rng.integers(0, len(y_test), len(y_test))
    bootstrap_y = y_test.to_numpy()[indices]
    bootstrap_probability = test_probabilities[indices]
    if len(np.unique(bootstrap_y)) < 2:
        continue
    bootstrap_auc.append(roc_auc_score(bootstrap_y, bootstrap_probability))

confidence_interval = np.quantile(bootstrap_auc, [0.025, 0.975])
print("Test AUC:", round(tuned_results["AUC"], 4))
print("95% bootstrap CI:", confidence_interval.round(4).tolist())


## 10. Model interpretability

Permutation importance measures the change in ROC-AUC when one original feature is shuffled. It provides a deployment-oriented view of important predictors.

In [ ]:
permutation = permutation_importance(
    selected_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": permutation.importances_mean,
    "Std": permutation.importances_std,
}).sort_values("Importance", ascending=False)

display(importance.head(15).round(4))

top = importance.head(15).iloc[::-1]
plt.figure(figsize=(10, 7))
plt.barh(top["Feature"], top["Importance"], xerr=top["Std"])
plt.xlabel("Decrease in ROC-AUC")
plt.title("Permutation Importance on Held-Out Test Set")
plt.tight_layout()
plt.show()


## 11. Save deployment artefacts

The saved object is the complete fitted pipeline, including preprocessing and the tuned model. The test-set export contains original feature values plus the target.

In [ ]:
MODEL_DIR = PROJECT_ROOT / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "notebooks" / "OSMI_Analysis_Results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(selected_model, MODEL_DIR / "best_model.joblib")

test_output = X_test.copy()
test_output["treatment"] = np.where(y_test == 1, "Yes", "No")
test_output.to_csv(PROCESSED_DIR / "test_set.csv", index=False)

comparison_table.round(6).to_csv(
    RESULTS_DIR / "model_comparison_cv.csv",
    index=False,
)
tuning_table.to_csv(
    RESULTS_DIR / "tuning_results.csv",
    index=False,
)

test_summary = pd.DataFrame([
    {"Model": selected_name, "Threshold": "0.50", **default_results},
    {"Model": selected_name, "Threshold": f"{best_threshold:.4f}", **tuned_results},
])
test_summary.to_csv(RESULTS_DIR / "test_results.csv", index=False)

def json_safe(value):
    if isinstance(value, dict):
        return {key: json_safe(item) for key, item in value.items()}
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value

metadata = {
    "model": selected_name,
    "scenario": "without_work_interfere",
    "features": FEATURES,
    "target": TARGET,
    "positive_class": "Yes",
    "best_params": json_safe(selected_search.best_params_),
    "probability_threshold": best_threshold,
    "test_auc": tuned_results["AUC"],
    "test_auc_ci95": confidence_interval.tolist(),
    "test_accuracy": tuned_results["Accuracy"],
    "test_f1": tuned_results["F1"],
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
}

with open(MODEL_DIR / "best_model_meta.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Saved pipeline:", MODEL_DIR / "best_model.joblib")
print("Saved metadata:", MODEL_DIR / "best_model_meta.json")
print("Saved test set:", PROCESSED_DIR / "test_set.csv")


## 12. Final summary

- Preprocessing remains inside the model pipeline to prevent data leakage.
- `work_interfere` is intentionally excluded from the fair final predictor.
- The test set is evaluated only after model and threshold selection.
- The saved model expects the 22 listed feature columns.
- This is a predictive association model, not a clinical diagnosis tool.